<a href="https://colab.research.google.com/github/EyalYe/ChampSim/blob/master/MLP_BNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# prefetch_train.py
# Training a binarized Mixer prefetcher with time-based split,
# proper LR scheduling, PC_UNK handling, clean windowing, robust checkpointing,
# and OOV-free targets via tail buckets + top-10 candidate

import os, math, json, warnings, random
import numpy as np
import pandas as pd
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.cluster import KMeans
from typing import Dict, Tuple, List

# Schedulers for warmup+cosine
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

# ────────────────────────────── CONFIG ──────────────────────────────
DATA_FILE      = "memory_accesses_pc_va.csv"
HISTORY_LENGTH = 64
CLUSTERS       = 8
TOP_PER_CL     = 2048   # per-cluster head vocab size (exact deltas)
BATCH_SIZE     = 256
NUM_EPOCHS     = 30
LEARNING_RATE  = 5e-4
VAL_FRACTION   = 0.30   # contiguous time split (global)
SEED           = 42
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PRINT_EVERY    = 1
TOPK_EVAL      = 10

# Model dims
PC_EMB         = 128
DELTA_EMB      = 64
CLUSTER_EMB    = 8
HIDDEN         = 256   # kept for compatibility in saved config, unused by Mixer
KERNEL         = 3     # kept for compatibility in saved config, unused by Mixer
DROPOUT        = 0.10

# Stabilization knobs
WARMUP_EPOCHS  = 8
WEIGHT_DECAY   = 1e-4
LABEL_SMOOTH   = 0.05

# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)

# Mixer-specific
MIXER_DIM         = PC_EMB + DELTA_EMB     # 192 with current settings
MIXER_DEPTH       = 3
TOKEN_MLP_DIM     = HISTORY_LENGTH
CHANNEL_MLP_DIM   = 2 * MIXER_DIM
SEMI_BINARY       = True

# OOV handling via tail buckets (sign × log2 magnitude)
USE_TAIL_BUCKETS      = True
TAIL_MAG_BINS         = 4                 # number of log2 magnitude bins
TAIL_BUCKETS          = 2 * TAIL_MAG_BINS  # + / - for each bin
BUCKET_EXPAND_TOPK    = 8                 # how many concrete deltas to expand per bucket
# Derived vocab sizes (inputs & outputs)
DELTA_INPUTS_PER_CLUSTER = TOP_PER_CL + (TAIL_BUCKETS if USE_TAIL_BUCKETS else 1)
N_CLASSES_PER_CLUSTER    = TOP_PER_CL + (TAIL_BUCKETS if USE_TAIL_BUCKETS else 0)

# EMA
EMA_DECAY         = 0.999
USE_EMA           = True

# Training stability toggles
CLIP_GRAD_NORM       = 0.5
DETERMINISTIC        = False
TRAIN_INV_ONLY       = True
USE_WEIGHTED_SAMPLER = False

# Determinism setup
if DETERMINISTIC:
    os.environ["PYTHONHASHSEED"] = "0"
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":16:8")

# ────────────────────────────── IO HELPERS ──────────────────────────
def read_table(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext in (".xlsx", ".xls"):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)
    needed = {"addr", "ip", "cache_hit"}
    missing = needed - set(map(str.lower, df.columns))
    if missing:
        df.columns = [c.lower() for c in df.columns]
        missing = needed - set(df.columns)
        if missing:
            raise ValueError(f"Missing required columns: {missing}")
    return df

def build_pc_vocab(train_df: pd.DataFrame) -> Dict[int, int]:
    """
    Map training PCs (train_df['pc_int']) to ids 1..N.
    Reserve 0 for UNK so downstream code can do: n_pcs_with_unk = len(pc2id) + 1
    """
    pcs = train_df["pc_int"].unique().tolist()
    pcs.sort()
    return {int(pc): i + 1 for i, pc in enumerate(pcs)}  # UNK = 0 (reserved)


# ─────────────────────── DATA PREPROCESS ──────────────────
def time_based_split(df: pd.DataFrame, val_fraction: float) -> Tuple[pd.DataFrame, pd.DataFrame]:
    n = len(df)
    cut = int((1.0 - val_fraction) * n)
    train_df = df.iloc[:cut].copy()
    val_df   = df.iloc[cut:].copy()
    return train_df, val_df

def fit_kmeans_on_train(train_blocks: np.ndarray, n_clusters: int) -> KMeans:
    uniq = np.unique(train_blocks).reshape(-1, 1)
    if len(uniq) < n_clusters:
        warnings.warn(f"[KMeans] Fewer unique blocks ({len(uniq)}) than clusters ({n_clusters}). Reducing clusters.")
        n_clusters = len(uniq)
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=SEED)
    kmeans.fit(uniq)
    return kmeans

def per_cluster_topk_deltas(train_df: pd.DataFrame, clusters: int, top_per_cl: int) -> Dict[int, Dict[int, int]]:
    """Stable head mapping: delta -> class_id [0..top_per_cl-1]."""
    out = {}
    for c in range(clusters):
        vals = train_df.loc[train_df["cluster"] == c, "local_delta"]
        if len(vals) == 0:
            out[c] = {}
            continue
        counts = vals.value_counts()
        dfc = pd.DataFrame({"delta": counts.index.astype(int), "cnt": counts.values})
        dfc.sort_values(["cnt", "delta"], ascending=[False, True], inplace=True, kind="stable")
        top = dfc["delta"].tolist()[:top_per_cl]
        out[c] = {int(d): i for i, d in enumerate(top)}
    return out

def invert_head_vocab(delta_out2id_per: Dict[int, Dict[int,int]]) -> Dict[int, Dict[int,int]]:
    """id -> delta, per cluster."""
    inv = {}
    for c, m in delta_out2id_per.items():
        inv[c] = {int(i): int(d) for d, i in m.items()}
    return inv

def add_block_and_pc_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if df["addr"].dtype != object: df["addr"] = df["addr"].astype(str)
    if df["ip"].dtype   != object: df["ip"]   = df["ip"].astype(str)
    df = df[df["cache_hit"] == 0].copy()
    df["block"] = df["addr"].apply(lambda x: int(str(x), 16))  # already aligned
    df["pc_int"] = df["ip"].apply(lambda x: int(str(x), 16))
    df.reset_index(drop=True, inplace=True)
    return df

def assign_clusters(df: pd.DataFrame, kmeans: KMeans) -> pd.DataFrame:
    df = df.copy()
    df["cluster"] = kmeans.predict(df["block"].values.reshape(-1, 1))
    return df

def compute_local_deltas_per_subset(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["local_delta"] = df.groupby("cluster")["block"].diff().fillna(0).astype(int)
    return df

# ─────────────── Tail bucketization utilities ────────────
def bucketize_delta(delta: int, mag_bins: int) -> int:
    """Return bucket id in [0, 2*mag_bins-1]: positives [0..mag_bins-1], negatives [mag_bins..2*mag_bins-1]."""
    sign = 0 if delta >= 0 else 1
    mag = abs(int(delta))
    if mag == 0:
        m = 0
    else:
        m = int(math.floor(math.log2(mag)))
        m = max(0, min(m, mag_bins - 1))
    return sign * mag_bins + m

def build_bucket_fallbacks(train_df: pd.DataFrame,
                           clusters: int,
                           head_map: Dict[int, Dict[int,int]],
                           mag_bins: int,
                           topm: int) -> Dict[int, Dict[int, List[int]]]:
    """
    For each cluster and each bucket, store top 'topm' frequent OOV deltas (not in head_map).
    Returns: per_cluster_bucket2list[cluster][bucket_id] = [delta1, delta2, ...]
    """
    per = {}
    for c in range(clusters):
        sub = train_df[train_df["cluster"] == c]
        head_set = set(head_map.get(c, {}).keys())
        oov = sub.loc[~sub["local_delta"].isin(head_set), "local_delta"]
        per_c = {b: [] for b in range(2 * mag_bins)}
        if len(oov) > 0:
            counts = oov.value_counts()
            dfb = pd.DataFrame({"delta": counts.index.astype(int), "cnt": counts.values})
            dfb["bucket"] = dfb["delta"].apply(lambda d: bucketize_delta(int(d), mag_bins))
            dfb.sort_values(["bucket", "cnt", "delta"], ascending=[True, False, True], inplace=True, kind="stable")
            for b in range(2 * mag_bins):
                top = dfb[dfb["bucket"] == b]["delta"].tolist()[:topm]
                per_c[b] = [int(x) for x in top]
        per[c] = per_c
    return per

def label_and_inputs_from_vocab(df: pd.DataFrame,
                                delta_out2id_per: Dict[int, Dict[int, int]],
                                top_per_cl: int,
                                pc2id: Dict[int, int],
                                use_tail_buckets: bool,
                                mag_bins: int) -> pd.DataFrame:
    """
    Create y (labels), delta_in_id (for embeddings), pc_id, and delta_global.
    With tail buckets: labels ∈ [0 .. TOP_PER_CL + TAIL_BUCKETS - 1] (no OOV).
    """
    df = df.copy()

    # labels + input ids
    def encode(row):
        c = int(row["cluster"]); d = int(row["local_delta"])
        head = delta_out2id_per.get(c, {})
        if d in head:
            lab = head[d]
            din = lab
        else:
            if use_tail_buckets:
                b = bucketize_delta(d, mag_bins)
                lab = top_per_cl + b
                din = top_per_cl + b
            else:
                lab = -1
                din = top_per_cl  # UNK
        return pd.Series((lab, din))

    df[["delta_out_id","delta_in_id"]] = df.apply(encode, axis=1)

    # pc ids: 0 is UNK
    df["pc_id"] = df["pc_int"].map(lambda x: pc2id.get(int(x), 0)).astype(int)

    # global delta embedding index
    df["delta_global"] = df["cluster"] * DELTA_INPUTS_PER_CLUSTER + df["delta_in_id"]
    return df

def build_windows(df: pd.DataFrame, history_len: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    windows = []
    for c, group in df.groupby("cluster", sort=True):
        pcs = group["pc_id"].values
        dgs = group["delta_global"].values
        y   = group["delta_out_id"].values
        for i in range(history_len, len(pcs)):
            windows.append((int(c), pcs[i-history_len:i], dgs[i-history_len:i], int(y[i])))
    if not windows:
        return (np.array([], dtype=np.int64),
                np.empty((0, history_len), dtype=np.int64),
                np.empty((0, history_len), dtype=np.int64),
                np.array([], dtype=np.int64))
    clusters, X_pc, X_dg, y = zip(*windows)
    return (np.array(clusters, dtype=np.int64),
            np.stack(X_pc).astype(np.int64),
            np.stack(X_dg).astype(np.int64),
            np.array(y, dtype=np.int64))

# ───────────────────────────── DATASET ──────────────────────────────
class PrefetchDataset(Dataset):
    def __init__(self, cl, pcs, dglob, labels):
        self.cl = torch.as_tensor(cl, dtype=torch.long)
        self.pcs = torch.as_tensor(pcs, dtype=torch.long)
        self.dglob = torch.as_tensor(dglob, dtype=torch.long)
        self.labels = torch.as_tensor(labels, dtype=torch.long)
    def __len__(self): return self.labels.shape[0]
    def __getitem__(self, idx):
        return (self.cl[idx], self.pcs[idx], self.dglob[idx]), self.labels[idx]

# ────────────────────────────── MODEL ───────────────────────────────
class BinaryLinear(nn.Module):
    def __init__(self, in_f, out_f, bias=False):
        super().__init__()
        self.W = nn.Parameter(torch.empty(out_f, in_f))
        nn.init.kaiming_uniform_(self.W, a=math.sqrt(5))
        self.bias = nn.Parameter(torch.zeros(out_f)) if bias else None
    def forward(self, x):
        W = self.W
        alpha = W.abs().mean(dim=1, keepdim=True)  # per-output
        Wb = alpha * torch.sign(W)
        Wb[torch.isnan(Wb)] = 0.0
        W_ste = W + (Wb - W).detach()
        return F.linear(x, W_ste, self.bias)

class PreSign(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.ln = nn.LayerNorm(dim)
    def forward(self, x):
        x = self.ln(x)
        y = x.sign()
        y[y == 0] = 1
        return x + (y - x).detach()

class MixerBlock(nn.Module):
    def __init__(self, seq_len, dim, token_mlp_dim, channel_mlp_dim, dropout=0.0, semi_binary=True):
        super().__init__()
        self.pre_tok = PreSign(dim)
        self.tok_fc1 = BinaryLinear(seq_len, token_mlp_dim, bias=False)
        self.tok_fc2 = (nn.Linear(token_mlp_dim, seq_len, bias=False)
                        if semi_binary else BinaryLinear(token_mlp_dim, seq_len, bias=False))
        self.pre_chn = PreSign(dim)
        self.chn_fc1 = BinaryLinear(dim, channel_mlp_dim, bias=False)
        self.chn_fc2 = (nn.Linear(channel_mlp_dim, dim, bias=False)
                        if semi_binary else BinaryLinear(channel_mlp_dim, dim, bias=False))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):  # x: (B, L, D)
        y = self.pre_tok(x)
        y = y.transpose(1, 2)
        y = self.tok_fc2(F.gelu(self.tok_fc1(y)))
        y = y.transpose(1, 2)
        x = x + self.drop(y)

        y = self.pre_chn(x)
        y = self.chn_fc2(F.gelu(self.chn_fc1(y)))
        x = x + self.drop(y)
        return x

class PrefetchBiMixer(nn.Module):
    def __init__(self, n_clusters, n_pcs_with_unk, n_delta_inputs_per_cluster,
                 pc_emb, delta_emb, cluster_emb,
                 dim, mixer_depth, token_mlp_dim, channel_mlp_dim,
                 dropout, n_classes_per_cluster, semi_binary, seq_len):
        super().__init__()
        self.n_clusters = n_clusters
        self.n_classes  = n_classes_per_cluster

        self.cluster_embed = nn.Embedding(n_clusters, cluster_emb)
        self.pc_embed      = nn.Embedding(n_pcs_with_unk, pc_emb)
        self.delta_embed   = nn.Embedding(n_clusters * n_delta_inputs_per_cluster, delta_emb)

        in_dim = pc_emb + delta_emb
        self.in_proj = nn.Linear(in_dim, dim, bias=False)

        self.blocks = nn.ModuleList([
            MixerBlock(seq_len=seq_len, dim=dim,
                       token_mlp_dim=token_mlp_dim,
                       channel_mlp_dim=channel_mlp_dim,
                       dropout=dropout,
                       semi_binary=semi_binary)
            for _ in range(mixer_depth)
        ])
        self.drop = nn.Dropout(dropout)

        self.heads = nn.ModuleList([
            nn.Linear(2*dim + cluster_emb, n_classes_per_cluster)
            for _ in range(n_clusters)
        ])

    def forward(self, cl_ids, pc_seq, dglob_seq):
        p = self.pc_embed(pc_seq)           # (B, L, pc_emb)
        d = self.delta_embed(dglob_seq)     # (B, L, delta_emb)
        x = torch.cat([p, d], dim=-1)       # (B, L, pc_emb+delta_emb)
        x = self.in_proj(x)                 # (B, L, dim)
        for blk in self.blocks:
            x = blk(x)                      # (B, L, dim)

        xT = x.transpose(1, 2)              # (B, dim, L)
        pooled_avg = xT.mean(dim=-1)        # (B, dim)
        pooled_max = xT.amax(dim=-1)        # (B, dim)

        pooled = torch.cat([pooled_avg, pooled_max], dim=1)  # (B, 2*dim)
        pooled = self.drop(pooled)

        c = self.cluster_embed(cl_ids)      # (B, cluster_emb)
        h = torch.cat([pooled, c], dim=1)   # (B, 2*dim + cluster_emb)

        B = h.size(0)
        out = torch.empty(B, self.n_classes, device=h.device)
        for k in range(self.n_clusters):
            m = (cl_ids == k)
            if m.any():
                out[m] = self.heads[k](h[m])
        return out

# ───────────────────────────── EMA ───────────────────────
class EMA:
    def __init__(self, model: nn.Module, decay: float):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}
        self.backup = None
    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            self.shadow[k].mul_((self.decay)).add_(v.detach(), alpha=1 - self.decay)
    @torch.no_grad()
    def apply(self, model: nn.Module):
        self.backup = {k: v.clone() for k, v in model.state_dict().items()}
        model.load_state_dict(self.shadow, strict=False)
    @torch.no_grad()
    def restore(self, model: nn.Module):
        model.load_state_dict(self.backup, strict=False)
        self.backup = None

# ─────────────────────────── EVALUATION ─────────────────────────────
@torch.no_grad()
def evaluate_prefetch(model, data_loader, device, topk=10):
    model.eval()
    total_samples   = 0
    correct_top1    = 0
    hits_topk       = 0
    tail_events     = 0  # labels in tail buckets (>= TOP_PER_CL)

    for (cl_b, pc_b, dg_b), y_b in data_loader:
        cl_b, pc_b, dg_b, y_b = cl_b.to(device), pc_b.to(device), dg_b.to(device), y_b.to(device)
        logits = model(cl_b, pc_b, dg_b)  # (B, C)
        B, C = logits.shape
        k_eff = min(topk, C)

        preds1 = logits.argmax(dim=1)
        correct_top1 += (preds1 == y_b).sum().item()

        topk_idx = logits.topk(k_eff, dim=1).indices  # (B, k_eff)
        hits_topk += (topk_idx == y_b.unsqueeze(1)).any(dim=1).sum().item()

        tail_events += (y_b >= TOP_PER_CL).sum().item()
        total_samples += B

    acc_top1 = (correct_top1 / total_samples) if total_samples else 0.0
    prec_atk = (hits_topk / total_samples)    if total_samples else 0.0
    tail_rate = (tail_events / total_samples) if total_samples else 0.0
    return acc_top1, prec_atk, tail_rate


# ────────── Utility: map logits to top-10 concrete deltas per sample ──────────
@torch.no_grad()
def logits_to_topk_deltas(logits: torch.Tensor,
                          cl_ids: torch.Tensor,
                          id2delta_per: Dict[int, Dict[int,int]],
                          bucket_fallbacks: Dict[int, Dict[int, List[int]]],
                          k: int = 10) -> List[List[int]]:
    """
    For each sample:
      - Head class (< TOP_PER_CL): map class_id -> exact delta
      - Bucket class (>= TOP_PER_CL): expand using bucket_fallbacks[cluster][bucket]
    Deduplicate, preserve order, and truncate to k.
    """
    logits = logits.detach()
    B, C = logits.shape
    top_idx = logits.topk(min(k, C), dim=1).indices.cpu().numpy()
    cl_np = cl_ids.cpu().numpy()
    out = []
    for i in range(B):
        c = int(cl_np[i])
        candidates = []
        seen = set()
        for cls in top_idx[i]:
            cls = int(cls)
            if cls < TOP_PER_CL:
                d = id2delta_per.get(c, {}).get(cls, None)
                if d is not None and d not in seen:
                    candidates.append(int(d)); seen.add(int(d))
            else:
                b = cls - TOP_PER_CL
                for d in bucket_fallbacks.get(c, {}).get(b, []):
                    if d not in seen:
                        candidates.append(int(d)); seen.add(int(d))
                    if len(candidates) >= k:
                        break
            if len(candidates) >= k:
                break
        out.append(candidates[:k])
    return out

# ─────────────────────────────── MAIN ───────────────────────────────
def main():
    # Load & normalize
    df = read_table(DATA_FILE)
    df = add_block_and_pc_columns(df)

    if len(df) < (HISTORY_LENGTH + 10):
        raise ValueError("Not enough miss events to form training windows. Add more data.")

    # Time-based split
    train_raw, val_raw = time_based_split(df, VAL_FRACTION)

    # Clusters from train only
    kmeans = fit_kmeans_on_train(train_raw["block"].to_numpy(), CLUSTERS)
    train_c = assign_clusters(train_raw, kmeans)
    val_c   = assign_clusters(val_raw,   kmeans)

    # Local deltas per subset
    train_c = compute_local_deltas_per_subset(train_c)
    val_c   = compute_local_deltas_per_subset(val_c)

    # Head vocab (train only) and its inverse
    delta_out2id_per = per_cluster_topk_deltas(train_c, CLUSTERS, TOP_PER_CL)
    id2delta_per     = invert_head_vocab(delta_out2id_per)

    # Build per-bucket fallback lists (train only; OOVs only)
    bucket_fallbacks = build_bucket_fallbacks(train_c, CLUSTERS, delta_out2id_per,
                                              TAIL_MAG_BINS, BUCKET_EXPAND_TOPK)

    # PC vocab (train only)
    pc2id = build_pc_vocab(train_c)
    n_pcs_with_unk = len(pc2id) + 1

    # Encode labels/inputs (with tail buckets)
    train_enc = label_and_inputs_from_vocab(train_c, delta_out2id_per, TOP_PER_CL, pc2id,
                                            USE_TAIL_BUCKETS, TAIL_MAG_BINS)
    val_enc   = label_and_inputs_from_vocab(val_c,   delta_out2id_per, TOP_PER_CL, pc2id,
                                            USE_TAIL_BUCKETS, TAIL_MAG_BINS)

    # Windows
    tr_clusters, Xpc_tr, Xdg_tr, y_tr = build_windows(train_enc, HISTORY_LENGTH)
    va_clusters, Xpc_va, Xdg_va, y_va = build_windows(val_enc,   HISTORY_LENGTH)

    if len(y_tr) == 0 or len(y_va) == 0:
        raise ValueError("No windows produced. Check HISTORY_LENGTH and data volume per cluster.")

    # Optional - filter train to in-vocab only
    if TRAIN_INV_ONLY:
        tr_mask = (y_tr >= 0)
        if tr_mask.sum() == 0:
            raise ValueError("All training windows invalid.")
        tr_clusters, Xpc_tr, Xdg_tr, y_tr = tr_clusters[tr_mask], Xpc_tr[tr_mask], Xdg_tr[tr_mask], y_tr[tr_mask]

    # DataLoaders
    g = torch.Generator(); g.manual_seed(SEED)
    train_ds = PrefetchDataset(tr_clusters, Xpc_tr, Xdg_tr, y_tr)
    val_ds   = PrefetchDataset(va_clusters, Xpc_va, Xdg_va, y_va)

    if USE_WEIGHTED_SAMPLER:
        unique, counts = np.unique(train_ds.cl.numpy(), return_counts=True)
        freq = {int(c): int(n) for c, n in zip(unique, counts)}
        weights = torch.tensor([1.0 / freq[int(c)] for c in train_ds.cl], dtype=torch.double)
        sampler = torch.utils.data.WeightedRandomSampler(weights, num_samples=len(weights),
                                                         replacement=True, generator=g)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                                  drop_last=False, num_workers=0, generator=g)
    else:
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  drop_last=False, num_workers=0, generator=g)

    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              drop_last=False, num_workers=0, generator=g)

    # Model
    model = PrefetchBiMixer(
        n_clusters=CLUSTERS,
        n_pcs_with_unk=n_pcs_with_unk,
        n_delta_inputs_per_cluster=DELTA_INPUTS_PER_CLUSTER,
        pc_emb=PC_EMB, delta_emb=DELTA_EMB, cluster_emb=CLUSTER_EMB,
        dim=MIXER_DIM, mixer_depth=MIXER_DEPTH,
        token_mlp_dim=TOKEN_MLP_DIM, channel_mlp_dim=CHANNEL_MLP_DIM,
        dropout=DROPOUT,
        n_classes_per_cluster=N_CLASSES_PER_CLUSTER,
        semi_binary=SEMI_BINARY,
        seq_len=HISTORY_LENGTH
    ).to(DEVICE)

    # Optimizer
    if WEIGHT_DECAY and WEIGHT_DECAY > 0:
        opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    else:
        opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=0.0)

    # LR schedule
    warmup = LinearLR(opt, start_factor=0.2, total_iters=WARMUP_EPOCHS)
    cosine = CosineAnnealingLR(opt, T_max=max(1, NUM_EPOCHS - WARMUP_EPOCHS),
                               eta_min=LEARNING_RATE * 0.1)
    sched  = SequentialLR(opt, [warmup, cosine], milestones=[WARMUP_EPOCHS])

    # EMA
    ema = EMA(model, EMA_DECAY) if USE_EMA else None

    # Checkpoint helper
    def build_ckpt(epoch: int):
        return {
            "state_dict": model.state_dict(),
            "config": {
                "CLUSTERS": CLUSTERS,
                "TOP_PER_CL": TOP_PER_CL,
                "HISTORY_LENGTH": HISTORY_LENGTH,
                "n_pcs_with_unk": n_pcs_with_unk,
                "n_delta_inputs_per_cluster": DELTA_INPUTS_PER_CLUSTER,

                "pc_emb": PC_EMB, "delta_emb": DELTA_EMB, "cluster_emb": CLUSTER_EMB,

                "model_type": "BiMixer",
                "mixer_dim": MIXER_DIM, "mixer_depth": MIXER_DEPTH,
                "token_mlp_dim": TOKEN_MLP_DIM, "channel_mlp_dim": CHANNEL_MLP_DIM,
                "semi_binary": SEMI_BINARY, "dropout": DROPOUT,

                "use_tail_buckets": USE_TAIL_BUCKETS,
                "tail_mag_bins": TAIL_MAG_BINS,
                "tail_buckets": TAIL_BUCKETS,
                "bucket_expand_topk": BUCKET_EXPAND_TOPK,
                "n_classes_per_cluster": N_CLASSES_PER_CLUSTER,
            },
            "kmeans_centers": np.asarray(kmeans.cluster_centers_, dtype=np.float32).ravel(),
            "pc2id": {int(k): int(v) for k, v in pc2id.items()},
            "delta_out2id_per": {int(c): {int(d): int(i) for d, i in m.items()} for c, m in delta_out2id_per.items()},
            "id2delta_per": {int(c): {int(i): int(d) for i, d in m.items()} for c, m in invert_head_vocab(delta_out2id_per).items()},
            "bucket_fallbacks": {int(c): {int(b): [int(d) for d in lst] for b, lst in bm.items()} for c, bm in bucket_fallbacks.items()},
            "meta": {"seed": SEED, "val_fraction": VAL_FRACTION, "epoch": epoch}
        }

    # Best-tracking
    best_tuple = (-1.0, -1.0, 1.0)
    best_path  = "prefetch_mixer_ckpt_best.pt"
    last_path  = "prefetch_mixer_ckpt_last.pt"

    # 11) Train
    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        tot, corr = 0, 0

        for (cl_b, pc_b, dg_b), y_b in train_loader:
            cl_b, pc_b, dg_b, y_b = cl_b.to(DEVICE), pc_b.to(DEVICE), dg_b.to(DEVICE), y_b.to(DEVICE)
            logits = model(cl_b, pc_b, dg_b)

            opt.zero_grad()
            loss = F.cross_entropy(logits, y_b, label_smoothing=LABEL_SMOOTH)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD_NORM)
            opt.step()
            if ema: ema.update(model)

            preds = logits.argmax(dim=1)
            corr  += (preds == y_b).sum().item()
            tot   += y_b.numel()

        train_top1 = (corr / tot) if tot else 0.0

        # Validation (EMA weights if enabled)
        if ema: ema.apply(model)
        acc_top1, prec_atk, tail_rate = evaluate_prefetch(model, val_loader, DEVICE, topk=TOPK_EVAL)
        if ema: ema.restore(model)

        if epoch % PRINT_EVERY == 0:
            curr_lr = opt.param_groups[0]["lr"]
            print(f"Epoch {epoch:02d} | LR: {curr_lr:.6g} | Train@1: {train_top1*100:6.2f}% "
                  f"| Val@1: {acc_top1*100:6.2f}% "
                  f"| P@{TOPK_EVAL}: {prec_atk*100:6.2f}% "
                  f"| TAIL: {tail_rate*100:6.2f}%")

        # Save "last" every epoch
        torch.save(build_ckpt(epoch), last_path)

        # Save "best" on improvement
        curr_tuple = (prec_atk, acc_top1, -tail_rate)
        if curr_tuple > best_tuple:
            best_tuple = curr_tuple
            torch.save(build_ckpt(epoch), best_path)
            print(f"✔ New best at epoch {epoch}: "
                  f"P@{TOPK_EVAL}={prec_atk*100:.2f}%, "
                  f"Val@1={acc_top1*100:.2f}%, "
                  f"TAIL={tail_rate*100:.2f}% → saved to {best_path}")

        # Step scheduler
        sched.step()

    # Final save
    final_path = "prefetch_mixer_ckpt.pt"
    torch.save(build_ckpt(NUM_EPOCHS), final_path)
    print(f"Saved final checkpoint to {final_path}")
    print(f"Best checkpoint: {best_path} | Last checkpoint: {last_path}")

if __name__ == "__main__":
    main()


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (7) found smaller than n_clusters (8). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Epoch 01 | LR: 0.0001 | Train@1:  26.57% | Val@1:  27.37% | P@10:  64.26% | TAIL:  31.25%
✔ New best at epoch 1: P@10=64.26%, Val@1=27.37%, TAIL=31.25% → saved to prefetch_mixer_ckpt_best.pt
Epoch 02 | LR: 0.00015 | Train@1:  34.30% | Val@1:  34.70% | P@10:  74.71% | TAIL:  31.25%
✔ New best at epoch 2: P@10=74.71%, Val@1=34.70%, TAIL=31.25% → saved to prefetch_mixer_ckpt_best.pt
Epoch 03 | LR: 0.0002 | Train@1:  44.84% | Val@1:  45.55% | P@10:  83.20% | TAIL:  31.25%
✔ New best at epoch 3: P@10=83.20%, Val@1=45.55%, TAIL=31.25% → saved to prefetch_mixer_ckpt_best.pt
Epoch 04 | LR: 0.00025 | Train@1:  50.09% | Val@1:  49.84% | P@10:  85.79% | TAIL:  31.25%
✔ New best at epoch 4: P@10=85.79%, Val@1=49.84%, TAIL=31.25% → saved to prefetch_mixer_ckpt_best.pt
Epoch 05 | LR: 0.0003 | Train@1:  52.57% | Val@1:  52.95% | P@10:  87.65% | TAIL:  31.25%
✔ New best at epoch 5: P@10=87.65%, Val@1=52.95%, TAIL=31.25% → saved to prefetch_mixer_ckpt_best.pt
Epoch 06 | LR: 0.00035 | Train@1:  54.27% |

/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:209: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Epoch 09 | LR: 0.0005 | Train@1:  57.71% | Val@1:  57.25% | P@10:  90.88% | TAIL:  31.25%
✔ New best at epoch 9: P@10=90.88%, Val@1=57.25%, TAIL=31.25% → saved to prefetch_mixer_ckpt_best.pt
Epoch 10 | LR: 0.00049771 | Train@1:  59.01% | Val@1:  58.55% | P@10:  91.41% | TAIL:  31.25%
✔ New best at epoch 10: P@10=91.41%, Val@1=58.55%, TAIL=31.25% → saved to prefetch_mixer_ckpt_best.pt
Epoch 11 | LR: 0.000490886 | Train@1:  60.11% | Val@1:  59.10% | P@10:  91.79% | TAIL:  31.25%
✔ New best at epoch 11: P@10=91.79%, Val@1=59.10%, TAIL=31.25% → saved to prefetch_mixer_ckpt_best.pt
Epoch 12 | LR: 0.000479667 | Train@1:  61.08% | Val@1:  59.86% | P@10:  92.07% | TAIL:  31.25%
✔ New best at epoch 12: P@10=92.07%, Val@1=59.86%, TAIL=31.25% → saved to prefetch_mixer_ckpt_best.pt
Epoch 13 | LR: 0.000464282 | Train@1:  61.89% | Val@1:  60.52% | P@10:  92.31% | TAIL:  31.25%
✔ New best at epoch 13: P@10=92.31%, Val@1=60.52%, TAIL=31.25% → saved to prefetch_mixer_ckpt_best.pt
Epoch 14 | LR: 0.00044